<a href="https://colab.research.google.com/github/Andrew-lee0615/value-at-risk-analysis/blob/main/05_backtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf


In [2]:

tickers = ["NVDA","INTC", "AMD", "MU", "MRVL", "WDC", "AVGO", "SPY"]
prices = yf.download(tickers, period="5y", auto_adjust=True)["Close"]
prices.tail()


[*********************100%***********************]  8 of 8 completed


Ticker,AMD,AVGO,INTC,MRVL,MU,NVDA,SPY,WDC
Date,,,,,,,,
2026-10-12,594.239990,354.730011,101.349998,265.859985,988.884338,226.320007,778.099976,386.940002
2026-10-13,605.530029,357.309998,102.809998,268.730011,953.040039,228.520004,778.010010,380.709991
2026-10-14,570.820007,346.250000,98.120003,245.279999,914.130005,221.300003,769.599976,364.100006
2026-10-15,562.229980,344.329987,95.370003,241.240005,931.239990,217.229996,771.789978,356.450012
2026-10-16,528.750000,328.750000,89.849998,227.830002,878.840027,214.130005,763.250000,332.000000


In [3]:
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.head()

Ticker,AMD,AVGO,INTC,MRVL,MU,NVDA,SPY,WDC
Date,,,,,,,,
2021-10-19,-0.000859,0.013869,0.013494,0.010073,0.004598,0.003055,0.007685,0.004502
2021-10-20,0.000516,-0.001961,0.002894,-0.009775,0.010306,-0.008425,0.003920,0.013564
2021-10-21,0.024946,0.009301,0.011314,-0.001788,0.005405,0.026299,0.002605,0.006890
2021-10-22,0.004098,0.002448,-0.124187,-0.011396,-0.016600,0.001497,-0.001037,0.004392
2021-10-25,0.020977,0.012839,-0.001011,0.020302,0.018346,0.019176,0.005349,-0.003864


In [4]:
stocks = ["NVDA","INTC","AMD","MU","MRVL","WDC","AVGO"]
port = log_returns[stocks].mean(axis=1)
port.head()

,0
Date,
2021-10-19,0.006962
2021-10-20,0.001017
2021-10-21,0.011767
2021-10-22,-0.019964
2021-10-25,0.012395


In [5]:
port_var_95 = -port.quantile(0.05)
port_var_99 = -port.quantile(0.01)
print(port_var_95)
print(port_var_99)


0.042301585872366584
0.06934890337007571


In [6]:
mu = port.mean()
sigma = port.std()

param_var_95 = 1.645 * sigma - mu
param_var_99 = 2.326 * sigma - mu
print(param_var_95)
print(param_var_99)

0.042850801837478865
0.061167476548703845


In [7]:
np.random.seed(1)
sims = np.random.normal(mu, sigma, 10000)

mc_var_95 = -np.percentile(sims, 5)
mc_var_99 = -np.percentile(sims, 1)
print(mc_var_95)
print(mc_var_99)

0.04216740947947374
0.060148957262298984


In [8]:
df = 4
scale = sigma / np.sqrt(df / (df - 2))

np.random.seed(1)
sims_t = mu + scale * np.random.standard_t(df, 10000)

mc_t_var_95 = -np.percentile(sims_t, 5)
mc_t_var_99 = -np.percentile(sims_t, 1)
print(mc_t_var_95)
print(mc_t_var_99)

0.03799685490488861
0.06904359566261088


In [9]:
compare = pd.DataFrame({
    "Historical": [port_var_95, port_var_99],
    "Parametric": [param_var_95, param_var_99],
    "MC Normal": [mc_var_95, mc_var_99],
    "MC t (df=4)": [mc_t_var_95, mc_t_var_99]
}, index=["95% VaR", "99% VaR"])
compare

,Historical,Parametric,MC Normal,MC t (df=4)
95% VaR,0.042302,0.042851,0.042167,0.037997
99% VaR,0.069349,0.061167,0.060149,0.069044


In [10]:
breach_95 = port < -port_var_95
breach_95.sum()

np.int64(63)

In [12]:
roll_var_95 = -port.rolling(250).quantile(0.05).shift(1)
bt = pd.DataFrame({"ret": port, "var95": roll_var_95}).dropna()
bt.head()

,ret,var95
Date,,
2022-10-17,0.024863,0.043361
2022-10-18,-0.006571,0.043361
2022-10-19,0.001746,0.043361
2022-10-20,0.003967,0.043361
2022-10-21,0.034172,0.043361


In [13]:
roll_breach_95 = (bt["ret"] < -bt["var95"]).sum()
expected_95 = len(bt) * 0.05
roll_breach_95, expected_95

(np.int64(62), 50.2)

In [14]:
bt["var99"] = -port.rolling(250).quantile(0.01).shift(1)
roll_breach_99 = (bt["ret"] < -bt["var99"]).sum()
expected_99 = len(bt) * 0.01
roll_breach_99, expected_99

(np.int64(14), 10.040000000000001)

In [16]:
mu_r = port.rolling(250).mean().shift(1)
sig_r = port.rolling(250).std().shift(1)

bt["param95"] = 1.645 * sig_r - mu_r
bt["param99"] = 2.326 * sig_r - mu_r
bt["t95"] = 1.507 * sig_r - mu_r
bt["t99"] = 2.649 * sig_r - mu_r
bt.head()

,ret,var95,var99,param95,param99,t95,t99
Date,,,,,,,
2022-10-17,0.024863,0.043361,0.06681,0.047141,0.065809,0.043359,0.074663
2022-10-18,-0.006571,0.043361,0.06681,0.047147,0.065847,0.043358,0.074716
2022-10-19,0.001746,0.043361,0.06681,0.047179,0.065879,0.043389,0.074748
2022-10-20,0.003967,0.043361,0.06681,0.047198,0.065889,0.043410,0.074754
2022-10-21,0.034172,0.043361,0.06681,0.047067,0.065744,0.043283,0.074603


In [17]:
models = ["var95", "param95", "t95", "var99", "param99", "t99"]
breaches = {}
for m in models:
    breaches[m] = (bt["ret"] < -bt[m]).sum()
breaches

{'var95': np.int64(62),
 'param95': np.int64(64),
 't95': np.int64(79),
 'var99': np.int64(14),
 'param99': np.int64(18),
 't99': np.int64(14)}

In [18]:
np.random.seed(1)
perfect = np.random.binomial(1004, 0.05, 10000)
perfect[:10]

array([52, 47, 53, 51, 54, 53, 46, 44, 54, 51])

In [19]:
((perfect >= 62) | (perfect <= 38)).mean()

np.float64(0.0913)

In [20]:
from scipy.stats import chi2

def kupiec(x, n, p):
    phat = x / n
    lr = -2 * ((n-x)*np.log(1-p) + x*np.log(p) - (n-x)*np.log(1-phat) - x*np.log(phat))
    pval = chi2.sf(lr, 1)
    return lr, pval

kupiec(62, len(bt), 0.05)

(np.float64(2.725390688937182), np.float64(0.09876395170185069))

In [21]:
ps = {"var95": 0.05, "param95": 0.05, "t95": 0.05,
      "var99": 0.01, "param99": 0.01, "t99": 0.01}

for m in models:
    lr, pval = kupiec(breaches[m], len(bt), ps[m])
    print(m, breaches[m], round(pval, 3), "Pass" if pval > 0.05 else "Reject")

var95 62 0.099 Pass
param95 64 0.055 Pass
t95 79 0.0 Reject
var99 14 0.236 Pass
param99 18 0.023 Reject
t99 14 0.236 Pass


Key Finding: I backtested Historical, Parametric (normal), and Student-t (df=4) VaR using a rolling 250-day window, so each day’s VaR used only past data (no look-ahead bias). Over 1,004 test days, I compared actual breaches with the expected number and applied the Kupiec test.
Only Historical VaR passed at both levels (62 breaches vs 50 expected at 95%, 14 vs 10 at 99%). Parametric VaR failed at 99% (18 breaches) because the normal distribution assumes thin tails. Student-t VaR failed at 95% (79 breaches) because, at the same volatility, it puts less weight in the middle range of losses. No single model fits every confidence level.